In [31]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, ContentFormat,DocumentAnalysisFeature
from azure.ai.documentintelligence.models import DocumentTable
from langchain.text_splitter import MarkdownHeaderTextSplitter
import os

from dotenv import load_dotenv
load_dotenv(override=True)

AZURE_DOC_INTELLIGENCE_ENDPOINT = os.environ["AZURE_DOC_INTELLIGENCE_ENDPOINT"]
AZURE_DOC_INTELLIGENCE_KEY = os.environ["AZURE_DOC_INTELLIGENCE_KEY"]

In [32]:
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

credential =AzureKeyCredential(AZURE_DOC_INTELLIGENCE_KEY)
document_intelligence_client = DocumentIntelligenceClient(
    endpoint=AZURE_DOC_INTELLIGENCE_ENDPOINT,
    credential=credential, api_version = "2024-11-30"
)

In [33]:
def text_html_processing(OcrExtractionDIOutput):
    offset = 0
    page_map = []
    page_map_dict =[]

    for page_num, page in enumerate(OcrExtractionDIOutput.pages):
        tables_on_page = [
            table
            for table in (OcrExtractionDIOutput.tables or [])
            if table.bounding_regions and table.bounding_regions[0].page_number == page_num + 1
        ]
        #print(tables_on_page)

        # mark all positions of the table spans in the page
        page_offset = page.spans[0].offset
        page_length = page.spans[0].length
        table_chars = [-1] * page_length
        for table_id, table in enumerate(tables_on_page):
            for span in table.spans:
                # replace all table spans with "table_id" in table_chars array
                for i in range(span.length):
                    idx = span.offset - page_offset + i
                    if idx >= 0 and idx < page_length:
                        table_chars[idx] = table_id

        # build page text by replacing characters in table spans with table html
        page_text = ""
        added_tables = set()
        for idx, table_id in enumerate(table_chars):
            if table_id == -1:
                page_text += OcrExtractionDIOutput.content[page_offset + idx]
            elif table_id not in added_tables:
                page_text += table_to_html(tables_on_page[table_id])
                added_tables.add(table_id)

        page_text += " "
        page_map.append((page_num+1, offset, page_text))

        single_page_dict = {}
        single_page_dict['page_num']= page_num+1
        single_page_dict['content'] = page_text
        single_page_dict['offset'] = offset
        page_map_dict.append(single_page_dict)

        offset += len(page_text)

    return page_map_dict

In [34]:
def OcrExtractionDI(relative_path: str, Markdown: [bool]=True):
    
    path_to_document = os.path.abspath(
        os.path.join(relative_path))
    
    if Markdown==True:
        output_format = ContentFormat.MARKDOWN
    else:
        output_format = None

    with open(path_to_document, "rb") as f:
        poller = document_intelligence_client.begin_analyze_document("prebuilt-layout", 
                                                                    analyze_request=f, content_type="application/octet-stream", 
                                                                    output_content_format=output_format)
    OcrExtractionDIOutput = poller.result()
    
    if Markdown==False:
        pagemap = text_html_processing(OcrExtractionDIOutput)
        extracted_processed_text = pagemap
    else:
        extracted_processed_text = OcrExtractionDIOutput

    return extracted_processed_text

#### .pdf processing

In [39]:
output_format = ContentFormat.MARKDOWN
path_to_document = "10K-MSFT-07-27-2023.pdf"

with open(path_to_document, "rb") as f:
    poller = document_intelligence_client.begin_analyze_document("prebuilt-layout", 
                                                                analyze_request=f, content_type="application/octet-stream", 
                                                                output_content_format=output_format)

In [48]:
OcrExtractionDIOutput = poller.result()
#print(OcrExtractionDIOutput.content)

#### .docx processing

In [45]:
output_format = ContentFormat.MARKDOWN
path_to_document = "10K-MSFT-07-27-2023.docx"

with open(path_to_document, "rb") as f:
    poller = document_intelligence_client.begin_analyze_document("prebuilt-layout", 
                                                                analyze_request=f, content_type="application/octet-stream", 
                                                                output_content_format=output_format)

In [47]:
OcrExtractionDIOutput = poller.result()
#print(OcrExtractionDIOutput.content)